# CDC Diabetes Health Indicators

**Clustering non supervisé des profils de santé et de mode de vie**

## Sommaire

1. **Rappel du contexte (BO et DSO)** — objectifs, données, exploration et préparation.
2. **Modélisation** — choix des paramètres, K-Means et DBSCAN.
3. **Évaluation** — métriques, comparaison, profils et interprétation.
4. **Déploiement** — sauvegarde du modèle, prédiction de nouveaux profils et utilisation.


# 1. Rappel du contexte (BO et DSO)

**Business Objective (BO).** Identifier des profils de santé et de mode de vie afin de mieux cibler les actions de prévention du diabète.

**Data Science Objective (DSO).** Regrouper les individus par clustering non supervisé, décrire les groupes obtenus à partir des variables de santé et de mode de vie, puis comparer K-Means et DBSCAN à l’aide de métriques adaptées.

Le statut diabétique est conservé uniquement pour une description a posteriori : il n’est pas utilisé pour former les clusters.


## 1.1 Préparer l’environnement

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import seaborn as sns
from IPython.display import display

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
sns.set_theme(style="whitegrid")


## 1.2 Charger les données

Le jeu de données utilisé est le jeu *CDC Diabetes Health Indicators* de l’UCI (ID 891).

In [ ]:
def load_cdc_diabetes():
    try:
        from ucimlrepo import fetch_ucirepo
    except ImportError:
        import subprocess
        import sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "ucimlrepo"])
        from ucimlrepo import fetch_ucirepo

    dataset = fetch_ucirepo(id=891)
    features = dataset.data.features.copy()
    target = dataset.data.targets.copy()
    data = pd.concat(
        [features.reset_index(drop=True), target.reset_index(drop=True)], axis=1
    )
    return data, "UCI-891"

df, source_used = load_cdc_diabetes()
df.columns = [str(col).strip() for col in df.columns]
print("Dimensions :", df.shape)
display(df.head())


## 1.3 Explorer les données

On vérifie les dimensions, les types de variables, les valeurs manquantes, les statistiques descriptives et les doublons.

In [ ]:
print("Nombre d’individus :", f"{df.shape[0]:,}")
print("Nombre de colonnes :", df.shape[1])
print("\nTypes des variables et valeurs non nulles :")
df.info()
print("\nValeurs manquantes par colonne (les plus importantes en premier) :")
missing_summary = pd.DataFrame({
    "Valeurs manquantes": df.isna().sum(),
    "Pourcentage (%)": (100 * df.isna().mean()).round(2),
})
display(missing_summary.sort_values("Valeurs manquantes", ascending=False).head(15))
print("Nombre de lignes entièrement dupliquées :", int(df.duplicated().sum()))


Les doublons exacts sont comptés et conservés : avec 21 variables principalement binaires ou ordinales, plusieurs personnes peuvent présenter exactement le même profil observé. Les supprimer modifierait les effectifs et la densité locale. Cette décision conserve les données d’origine, mais les doublons peuvent augmenter la densité perçue par DBSCAN et donc influencer ses groupes.

### 1.3.1 Séparer le statut diabétique

La variable de statut sera utilisée uniquement pour décrire les groupes après le clustering ; elle est exclue des variables d’apprentissage.

In [ ]:
target_candidates = ["Diabetes_binary", "Diabetes_012", "Diabetes"]
target_col = next((col for col in target_candidates if col in df.columns), None)
id_candidates = ["ID", "id", "PatientID", "patient_id"]
id_cols = [col for col in id_candidates if col in df.columns]

if target_col:
    display(df[target_col].value_counts(dropna=False).sort_index().to_frame("effectif"))

In [ ]:
if target_col:
    df[target_col].value_counts(normalize=True, dropna=False).sort_index().mul(100).plot(
        kind="bar", figsize=(7, 4), rot=0
    )
    plt.title("Distribution du statut diabétique (description seulement)")
    plt.xlabel("Valeur de la cible")
    plt.ylabel("Pourcentage (%)")
    plt.tight_layout()
    plt.show()

**Interprétation.** Ce graphique décrit la répartition de la variable cible dans le jeu de données. Il sert de contexte et ne participe pas à la construction des groupes.

## 1.4 Sélection et préparation des variables

Les variables de santé et de mode de vie disponibles sont retenues. Le statut diabétique et les identifiants éventuels sont exclus du clustering.

In [ ]:
preferred_features = [
    "HighBP", "HighChol", "CholCheck", "BMI", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "GenHlth",
    "MentHlth", "PhysHlth", "DiffWalk", "Sex", "Age", "Education", "Income",
]
excluded_cols = list(id_cols)
if target_col:
    excluded_cols.append(target_col)

selected_features = [col for col in preferred_features if col in df.columns]
missing_preferred_features = [col for col in preferred_features if col not in df.columns]
if missing_preferred_features:
    print("Variables non trouvées dans cette version :", missing_preferred_features)

if not selected_features:
    selected_features = [
        col for col in df.select_dtypes(include=np.number).columns
        if col not in excluded_cols
    ]
if not selected_features:
    raise ValueError("Aucune variable numérique disponible pour le clustering.")

X = df[selected_features].copy()
print("Nombre de variables :", len(selected_features))


### 1.4.1 Distributions initiales

Les graphiques explorent la distribution de l’IMC et de quelques variables de santé avant la modélisation.

In [ ]:
# Histogramme et boxplot de BMI pour comprendre sa distribution
if "BMI" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    bmi_values = pd.to_numeric(df["BMI"], errors="coerce").dropna()
    axes[0].hist(bmi_values, bins=30)
    axes[0].set_title("Distribution de BMI")
    axes[0].set_xlabel("BMI")
    axes[0].set_ylabel("Effectif")

    sns.boxplot(x=bmi_values, ax=axes[1])
    axes[1].set_title("Boxplot de BMI")
    axes[1].set_xlabel("BMI")
    plt.tight_layout()
    plt.show()

# Comptage de quelques variables binaires souvent utiles pour décrire les profils
binary_to_plot = [c for c in ["HighBP", "HighChol", "Smoker", "PhysActivity"] if c in df.columns]
if binary_to_plot:
    fig, axes = plt.subplots(1, len(binary_to_plot), figsize=(4 * len(binary_to_plot), 3.5))
    axes = np.atleast_1d(axes)
    for ax, col in zip(axes, binary_to_plot):
        df[col].value_counts(dropna=False).sort_index().plot(kind="bar", ax=ax)
        ax.set_title(f"Répartition de {col}")
        ax.set_xlabel(col)
        ax.set_ylabel("Effectif")
        ax.tick_params(axis="x", rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("Aucune des variables binaires suggérées n’est présente dans ce fichier.")

**Interprétation.** L’histogramme et le boxplot résument la distribution de l’IMC et rendent visibles son étalement et ses valeurs extrêmes éventuelles, qui pèseront sur les distances du clustering. Les barres des variables binaires indiquent les effectifs de chaque modalité, utiles pour contextualiser les profils.

### 1.4.2 Valeurs manquantes et imputation

Les variables sont converties en numérique. Les valeurs infinies sont considérées comme manquantes ; l’IMC est imputé par la médiane et les autres variables par le mode.

In [ ]:
X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
missing_before = int(X.isna().sum().sum())
print("Valeurs manquantes avant traitement :", missing_before)

# Mémoriser les mêmes statistiques d’imputation pour la phase de déploiement.
fill_values = {}
for col in X.columns:
    observed = X[col].dropna()
    if col == "BMI" and not observed.empty:
        fill_value = float(observed.median())
    else:
        modes = observed.mode(dropna=True)
        fill_value = float(modes.iloc[0]) if len(modes) else 0.0
    fill_values[col] = fill_value
    X[col] = X[col].fillna(fill_value)

print("Valeurs manquantes après traitement :", int(X.isna().sum().sum()))

### 1.4.3 Corrélations

La matrice de corrélation permet d’observer les associations linéaires entre les variables numériques sélectionnées.

In [ ]:
corr_matrix = X.corr(numeric_only=True)
plt.figure(figsize=(12, 9))
sns.heatmap(corr_matrix, cmap="coolwarm", center=0, vmin=-1, vmax=1, linewidths=0.15)
plt.title("Matrice de corrélation des variables sélectionnées")
plt.xlabel("Variable")
plt.ylabel("Variable")
plt.tight_layout()
plt.show()

**Interprétation.** Les corrélations proches de +1 indiquent une association linéaire positive, celles proches de −1 une association négative, et celles proches de 0 une faible association linéaire. Une corrélation ne démontre pas un lien de causalité.

### 1.4.4 Standardisation

La standardisation place les variables sur une échelle comparable avant le calcul des distances. Les valeurs d’origine restent disponibles pour décrire les groupes.

In [ ]:
if X.isna().any().any():
    raise ValueError("Il reste des valeurs manquantes après le nettoyage.")

scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index,
)

print("Dimensions des données préparées :", X_scaled.shape)
display(X_scaled.describe().T[["mean", "std"]].round(2))


## 1.5 Échantillonnage et tableau récapitulatif

Pour garder un temps de calcul raisonnable, le clustering est réalisé sur un échantillon aléatoire reproductible de 3 000 personnes au maximum.

In [ ]:
CLUSTER_SAMPLE_SIZE = min(3000, len(X_scaled))
X_cluster = X_scaled.sample(n=CLUSTER_SAMPLE_SIZE, random_state=RANDOM_STATE).copy()
df_cluster_source = df.loc[X_cluster.index].copy()

if len(X_cluster) < 3:
    raise ValueError("Il faut au moins 3 individus pour le clustering.")

print(f"Échantillon utilisé : {len(X_cluster):,} personnes sur {len(X_scaled):,}.")

summary_rows = [
    ("Jeu de données", "CDC Diabetes Health Indicators — UCI, ID 891"),
    ("Nombre de lignes dans la source", f"{len(df):,}"),
    ("Nombre de colonnes dans la source", f"{df.shape[1]:,}"),
    ("Variables retenues", f"{len(selected_features)} : " + ", ".join(selected_features)),
    ("Traitement des valeurs manquantes", f"{missing_before} valeur(s) manquante(s) avant imputation dans les variables retenues ; {int(X.isna().sum().sum())} après. BMI : médiane ; autres variables : mode."),
    ("Taille de l’échantillon", f"{len(X_cluster):,} personne(s) sur {len(X_scaled):,} (maximum prévu : 3 000 ; RANDOM_STATE = {RANDOM_STATE})."),
]
dataset_summary = pd.DataFrame(summary_rows, columns=["Élément", "Valeur"])
pd.set_option("display.max_colwidth", None)
display(dataset_summary)

# 2. Modélisation

Deux approches de clustering non supervisé sont comparées sur les mêmes variables standardisées et le même échantillon : K-Means, qui construit des groupes autour de centroïdes, et DBSCAN, qui recherche des zones denses.

## 2.1 K-Means — choix du nombre de groupes

On compare plusieurs valeurs de `k` à l’aide de l’inertie (méthode du coude) et de la silhouette. Le choix reste un compromis entre la qualité de séparation et la facilité d’interprétation.

In [ ]:
k_values = list(range(2, min(7, len(X_cluster))))
inertias = []
silhouettes = []
SILHOUETTE_SAMPLE_SIZE = min(1500, len(X_cluster))

for k in k_values:
    candidate_model = KMeans(
        n_clusters=k,
        init="k-means++",
        random_state=RANDOM_STATE,
        n_init=10,
        max_iter=300,
    )
    candidate_labels = candidate_model.fit_predict(X_cluster)
    inertias.append(candidate_model.inertia_)
    silhouettes.append(
        silhouette_score(
            X_cluster,
            candidate_labels,
            sample_size=SILHOUETTE_SAMPLE_SIZE,
            random_state=RANDOM_STATE,
        )
    )

k_scores = pd.DataFrame({
    "k": k_values,
    "Inertie K-Means": inertias,
    "Silhouette": silhouettes,
})
display(k_scores.round(4))


Les deux graphiques sont complémentaires : la courbe d’inertie décrit la baisse de l’inertie quand `k` augmente, tandis que la silhouette mesure la cohésion et la séparation des groupes. On examine la présence éventuelle d’un coude sans présumer qu’il est net.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(k_values, inertias, marker="o")
axes[0].set_title("Méthode du coude — K-Means")
axes[0].set_xlabel("Nombre de clusters (k)")
axes[0].set_ylabel("Inertie")
axes[1].plot(k_values, silhouettes, marker="o")
axes[1].set_title("Silhouette selon k")
axes[1].set_xlabel("Nombre de clusters (k)")
axes[1].set_ylabel("Silhouette")
plt.tight_layout()
plt.show()

best_k = int(k_scores.loc[k_scores["Silhouette"].idxmax(), "k"])
print("Valeur de k proposée par la meilleure silhouette :", best_k)


**Interprétation.** La courbe d’inertie ne montre pas de coude net et incontestable sur l’intervalle testé. La valeur `k = 3` est donc retenue parce qu’elle maximise la silhouette parmi les valeurs examinées, et non parce qu’un coude évident aurait été identifié. Les scores restent modestes : la séparation entre groupes est limitée et la structure doit être interprétée comme exploratoire.

## 2.2 K-Means — principe et hyperparamètres retenus

**Principe.** K-Means attribue chaque individu au centroïde le plus proche, puis recalcule les centroïdes jusqu’à stabilisation ou atteinte du nombre maximal d’itérations.

**Étapes de l’algorithme :** (1) initialiser `k` centroïdes ; (2) affecter chaque individu au centroïde le plus proche ; (3) recalculer chaque centroïde comme la moyenne des individus affectés ; (4) répéter jusqu’à convergence ou jusqu’à `max_iter`.

**Avantages :** algorithme simple, rapide sur un échantillon de taille modérée et facile à résumer par des profils moyens. **Limites :** `k` doit être fixé à l’avance ; le résultat dépend de l’initialisation et peut être influencé par la géométrie des groupes et les valeurs extrêmes.

Le nombre de groupes retenu est choisi selon la silhouette parmi les valeurs testées ; la courbe du coude reste un repère visuel complémentaire. `init='k-means++'` est explicitement utilisé pour initialiser les centroïdes. `n_init=10` et `max_iter=300` sont des valeurs standard fixées pour ce travail, **non optimisées par une recherche d’hyperparamètres**. Le tableau récapitulatif de la section 2.6 affiche les valeurs réellement utilisées, leur justification et la méthode de choix.

In [ ]:
kmeans_model = KMeans(
    n_clusters=best_k,
    init="k-means++",
    random_state=RANDOM_STATE,
    n_init=10,
    max_iter=300,
)
labels_kmeans = kmeans_model.fit_predict(X_cluster)
print("Inertie K-Means :", round(float(kmeans_model.inertia_), 2))


## 2.3 Visualiser les groupes K-Means

Le clustering est appris sur les 21 variables standardisées. Pour faciliter l’interprétation métier, ce graphique représente les individus selon deux variables d’origine : la classe d’âge (`Age`) et l’IMC (`BMI`). Cette visualisation ne modifie pas les données utilisées pour entraîner le modèle.

In [ ]:
# Visualisation sur deux variables d’origine ; le modèle a été ajusté sur toutes les variables.
visualization_features = ["Age", "BMI"]
visualization_data = X.loc[X_cluster.index, visualization_features].copy()

# Léger jitter horizontal pour distinguer les observations qui ont le même code Age.
# Cette colonne sert uniquement au tracé et n'entre jamais dans le clustering.
jitter_rng = np.random.default_rng(RANDOM_STATE)
visualization_data["Age_jitter"] = visualization_data["Age"].to_numpy() + jitter_rng.uniform(
    low=-0.12, high=0.12, size=len(visualization_data)
)

kmeans_viz = visualization_data.copy()
kmeans_viz["Cluster"] = labels_kmeans.astype(str)

plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=kmeans_viz,
    x="Age_jitter",
    y="BMI",
    hue="Cluster",
    palette="tab10",
    s=24,
    alpha=0.55,
)
plt.title(f"K-Means — IMC selon la classe d’âge (k={best_k})")
plt.xlabel("Classe d’âge (code Age : 1 à 13 ; léger jitter visuel)")
plt.ylabel("IMC (variable BMI, unité du jeu de données)")
plt.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

# Effectif de chaque groupe juste après le graphique.
km_counts = pd.Series(labels_kmeans).value_counts().sort_index()
km_size_table = pd.DataFrame({
    "Effectif": km_counts,
    "Pourcentage (%)": (100 * km_counts / len(labels_kmeans)).round(2),
})
display(km_size_table)

**Interprétation.** Chaque couleur représente un groupe K-Means selon la classe d’âge et l’IMC. Cette vue permet de comparer visuellement les profils sur deux variables interprétables, mais les groupes sont formés à partir de l’ensemble des 21 variables standardisées.


## 2.4 DBSCAN — recherche de `eps` et modélisation

La courbe des k-distances sert à définir une plage de valeurs candidates pour `eps`. Les candidats sont ensuite comparés avec `min_samples` égal à 10, 22 ou 42. Pendant cette recherche, la silhouette hors bruit est estimée sur un sous-échantillon pour réduire le coût de calcul ; la comparaison finale des deux modèles utilise une silhouette exacte sur toutes les observations évaluables de l’échantillon de 3 000 personnes.

**Principe.** DBSCAN identifie des zones denses à partir de `eps` (rayon du voisinage) et de `min_samples` (nombre minimal de points dans le voisinage, point courant inclus). Les points atteignables depuis les points noyaux forment des clusters ; les points non rattachés reçoivent l’étiquette `-1` (bruit).

**Étapes :** (1) examiner le voisinage des points ; (2) repérer les points noyaux ; (3) étendre les clusters depuis ces noyaux ; (4) laisser en bruit les observations non rattachées.

**Avantages :** le nombre de groupes n’est pas fixé à l’avance et des groupes de formes variées peuvent être détectés. **Limites :** les résultats dépendent de `eps` et de `min_samples`, les densités hétérogènes sont difficiles à gérer et les métriques doivent expliciter le traitement du bruit.

In [ ]:
# Comparer plusieurs valeurs de min_samples et des candidats eps tirés de leurs courbes de k-distances.
MIN_SAMPLES_CANDIDATES = sorted({min(value, len(X_cluster)) for value in [10, 22, 42]})
eps_quantiles = [0.60, 0.70, 0.75, 0.80, 0.85, 0.90, 0.925, 0.95, 0.975]
k_distance_curves = {}
candidate_rows = []

for min_samples_value in MIN_SAMPLES_CANDIDATES:
    neighbors = NearestNeighbors(
        n_neighbors=min_samples_value,
        metric="euclidean",
        n_jobs=-1,
    )
    neighbors.fit(X_cluster)
    distances, _ = neighbors.kneighbors(X_cluster)
    k_distances_for_value = np.sort(distances[:, -1])
    k_distance_curves[min_samples_value] = k_distances_for_value
    eps_values = sorted(set(float(v) for v in np.quantile(k_distances_for_value, eps_quantiles)))

    for eps_value in eps_values:
        candidate_model = DBSCAN(
            eps=eps_value,
            min_samples=min_samples_value,
            metric="euclidean",
            n_jobs=-1,
        )
        candidate_labels = candidate_model.fit_predict(X_cluster)
        non_noise_mask = candidate_labels != -1
        n_clusters = len(set(candidate_labels) - {-1})
        n_non_noise = int(non_noise_mask.sum())
        noise_percent = 100 * float((candidate_labels == -1).mean())
        silhouette_non_noise = np.nan

        # La silhouette est calculée uniquement hors bruit et si au moins deux groupes sont présents.
        if n_clusters >= 2 and n_non_noise >= max(30, int(0.25 * len(X_cluster))):
            labels_kept = candidate_labels[non_noise_mask]
            if 1 < len(np.unique(labels_kept)) < len(labels_kept):
                silhouette_non_noise = silhouette_score(
                    X_cluster.iloc[np.flatnonzero(non_noise_mask)],
                    labels_kept,
                    sample_size=min(1000, n_non_noise),
                    random_state=RANDOM_STATE,
                )

        clustered_counts = pd.Series(candidate_labels[candidate_labels != -1]).value_counts()
        largest_cluster_share = (
            100 * float(clustered_counts.max()) / len(candidate_labels)
            if len(clustered_counts) else 0.0
        )
        candidate_rows.append({
            "min_samples": int(min_samples_value),
            "eps": eps_value,
            "Nombre de clusters (hors bruit)": n_clusters,
            "Individus non-bruit": n_non_noise,
            "Bruit (%)": noise_percent,
            "Part du plus grand cluster (%)": largest_cluster_share,
            "Silhouette (hors bruit)": silhouette_non_noise,
        })

dbscan_candidates = pd.DataFrame(candidate_rows)
print("Comparaison des candidats DBSCAN (la silhouette est calculée hors bruit) :")
display(dbscan_candidates.round(4))

# Résumé du meilleur candidat évalué pour chaque valeur de min_samples.
min_samples_comparison = (
    dbscan_candidates[dbscan_candidates["Silhouette (hors bruit)"].notna()]
    .sort_values(
        ["min_samples", "Silhouette (hors bruit)", "Bruit (%)"],
        ascending=[True, False, True],
    )
    .groupby("min_samples", as_index=False)
    .head(1)
    .sort_values("min_samples")
)
print("Meilleur candidat évalué pour chaque valeur de min_samples testée :")
display(min_samples_comparison.round(4))

# Règles de recevabilité : au moins deux groupes, couverture suffisante et au plus 50 % de bruit (même seuil que la sélection finale).
valid_candidates = dbscan_candidates[
    (dbscan_candidates["Nombre de clusters (hors bruit)"] >= 2)
    & (dbscan_candidates["Individus non-bruit"] >= max(30, int(0.30 * len(X_cluster))))
    & (dbscan_candidates["Bruit (%)"] <= 50)
    & dbscan_candidates["Silhouette (hors bruit)"].notna()
]

if not valid_candidates.empty:
    chosen_candidate = valid_candidates.sort_values(
        ["Silhouette (hors bruit)", "Bruit (%)", "Individus non-bruit"],
        ascending=[False, True, False],
    ).iloc[0]
    print("Réglage choisi selon la silhouette parmi les candidats respectant les critères.")
else:
    evaluable_candidates = dbscan_candidates[
        (dbscan_candidates["Nombre de clusters (hors bruit)"] >= 2)
        & dbscan_candidates["Silhouette (hors bruit)"].notna()
    ]
    if not evaluable_candidates.empty:
        chosen_candidate = evaluable_candidates.sort_values(
            ["Silhouette (hors bruit)", "Bruit (%)", "Individus non-bruit"],
            ascending=[False, True, False],
        ).iloc[0]
        print("Aucun candidat ne respecte tous les critères ; choix parmi les candidats évaluables.")
    else:
        chosen_candidate = dbscan_candidates.sort_values(
            ["Bruit (%)", "Individus non-bruit"],
            ascending=[True, False],
        ).iloc[0]
        print("Aucun candidat n’a une silhouette calculable avec au moins deux groupes ; le réglage le moins bruité est retenu à titre de repli.")

MIN_SAMPLES_DBSCAN = int(chosen_candidate["min_samples"])
dbscan_eps = float(chosen_candidate["eps"])
k_distances = k_distance_curves[MIN_SAMPLES_DBSCAN]

# Montrer la courbe de k-distances correspondant au min_samples effectivement retenu.
plt.figure(figsize=(9, 4))
plt.plot(k_distances)
plt.title(f"Distance au voisin de rang {MIN_SAMPLES_DBSCAN} (point courant inclus) — choix de eps")
plt.xlabel("Observations triées par distance")
plt.ylabel("Distance euclidienne au voisin choisi (variables standardisées)")
plt.tight_layout()
plt.show()

print(f"Paramètres DBSCAN retenus : eps={dbscan_eps:.4f}, min_samples={MIN_SAMPLES_DBSCAN}")
dbscan_model = DBSCAN(
    eps=dbscan_eps,
    min_samples=MIN_SAMPLES_DBSCAN,
    metric="euclidean",
    n_jobs=-1,
)
labels_dbscan = dbscan_model.fit_predict(X_cluster)
n_dbscan_clusters = len(set(labels_dbscan) - {-1})
dbscan_noise_percent = 100 * float(np.mean(labels_dbscan == -1))
print(f"Nombre de clusters hors bruit : {n_dbscan_clusters}")
print(f"Individus classés comme bruit : {dbscan_noise_percent:.2f} %")
if n_dbscan_clusters < 2:
    print("Attention : DBSCAN n’a pas trouvé au moins deux clusters. Les métriques de séparation seront indisponibles ; ce résultat limite l’utilité de cette configuration sur cet échantillon.")

Les valeurs `min_samples = 10`, `22` et `42` sont comparées. Pour chaque valeur, les candidats `eps` proviennent de quantiles de la courbe des k-distances correspondante. Le choix final privilégie la silhouette hors bruit parmi les configurations ayant au moins deux groupes, une couverture suffisante et un taux de bruit limité ; si aucune configuration ne remplit ces critères, un repli explicite est signalé. Les valeurs réellement retenues et leur justification figurent dans le tableau dynamique des hyperparamètres en section 2.6.

**Lecture de la courbe.** La courbe des k-distances sert de repère visuel pour `eps`, mais ne détermine pas automatiquement une valeur optimale. Le choix final dépend aussi des résultats comparatifs et du taux de bruit.

## 2.5 Visualiser les groupes DBSCAN

Le graphique utilise exactement les mêmes variables d’origine (`Age` et `BMI`), le même léger jitter visuel et le même échantillon que K-Means. Les observations classées comme bruit (`-1`) apparaissent en gris. Les clusters sont formés sur les 21 variables standardisées, pas uniquement sur les deux variables affichées.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
dbscan_viz = visualization_data.copy()
dbscan_viz["Cluster"] = labels_dbscan
labels_unique = sorted(np.unique(labels_dbscan))

noise_mask = dbscan_viz["Cluster"].to_numpy() == -1
if noise_mask.any():
    ax.scatter(
        dbscan_viz.loc[noise_mask, "Age_jitter"],
        dbscan_viz.loc[noise_mask, "BMI"],
        color="lightgrey", s=20, alpha=0.6, label="Bruit (-1)"
    )

cluster_ids = [label for label in labels_unique if label != -1]
colors = sns.color_palette("tab10", n_colors=max(1, len(cluster_ids)))
for cluster_id, color in zip(cluster_ids, colors):
    mask = dbscan_viz["Cluster"].to_numpy() == cluster_id
    ax.scatter(
        dbscan_viz.loc[mask, "Age_jitter"],
        dbscan_viz.loc[mask, "BMI"],
        color=color, s=24, alpha=0.7, label=f"Cluster {cluster_id}"
    )

ax.set_title(f"DBSCAN — IMC selon la classe d’âge (eps={dbscan_eps:.3f}, min_samples={MIN_SAMPLES_DBSCAN})")
ax.set_xlabel("Classe d’âge (code Age : 1 à 13 ; léger jitter visuel)")
ax.set_ylabel("IMC (variable BMI, unité du jeu de données)")
ax.legend(title="Groupe", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

print(f"Clusters détectés (hors bruit) : {n_dbscan_clusters}")
print(f"Observations classées comme bruit : {dbscan_noise_percent:.1f} %")

**Interprétation.** DBSCAN peut laisser certaines observations en bruit (`-1`) si elles ne sont pas dans une zone suffisamment dense selon les paramètres retenus. Cette étiquette décrit la densité dans l’espace des variables ; elle ne signifie pas qu’une personne est médicalement anormale.

In [ ]:
dbscan_cluster_counts = pd.Series(labels_dbscan).value_counts().sort_index()
dbscan_cluster_summary = pd.DataFrame({
    "Étiquette DBSCAN": dbscan_cluster_counts.index,
    "Effectif": dbscan_cluster_counts.values,
    "Pourcentage (%)": (100 * dbscan_cluster_counts.values / len(labels_dbscan)).round(2),
})
display(dbscan_cluster_summary)
print("L’étiquette -1 représente le bruit, pas un cluster de profil ordinaire.")

Le tableau de répartition permet de distinguer les groupes de DBSCAN des observations étiquetées comme bruit ; le bruit n’est pas interprété comme un groupe de profil ordinaire.

## 2.6 Récapitulatif des hyperparamètres

In [ ]:
hyperparams = {
    "K-Means": {
        "n_clusters": int(best_k),
        "init": "k-means++",
        "n_init": 10,
        "max_iter": 300,
        "random_state": RANDOM_STATE,
    },
    "DBSCAN": {
        "eps": float(dbscan_eps),
        "min_samples": int(MIN_SAMPLES_DBSCAN),
        "metric": "euclidean",
    },
}
print("Hyperparamètres finaux effectivement utilisés par les modèles :")
hyperparams_rows = [
    {"Modèle": model_name, "Paramètre": parameter, "Valeur retenue": value}
    for model_name, values in hyperparams.items()
    for parameter, value in values.items()
]
display(pd.DataFrame(hyperparams_rows))

hyperparameter_rationale = pd.DataFrame([
    {
        "Modèle": "K-Means", "Hyperparamètre": "n_clusters",
        "Valeur choisie": hyperparams["K-Means"]["n_clusters"],
        "Raison": "Nombre de groupes retenu selon la meilleure silhouette parmi les valeurs testées.",
        "Méthode de choix": "Silhouette ; lecture complémentaire de la courbe du coude, sans supposer qu’un coude net existe.",
    },
    {
        "Modèle": "K-Means", "Hyperparamètre": "init",
        "Valeur choisie": hyperparams["K-Means"]["init"],
        "Raison": "Initialisation k-means++ explicite, conforme à l’initialisation standard retenue.",
        "Méthode de choix": "Valeur standard fixée, non optimisée.",
    },
    {
        "Modèle": "K-Means", "Hyperparamètre": "n_init",
        "Valeur choisie": hyperparams["K-Means"]["n_init"],
        "Raison": "Plusieurs initialisations pour limiter la dépendance à un seul départ.",
        "Méthode de choix": "Valeur standard fixée, non optimisée par recherche d’hyperparamètres.",
    },
    {
        "Modèle": "K-Means", "Hyperparamètre": "max_iter",
        "Valeur choisie": hyperparams["K-Means"]["max_iter"],
        "Raison": "Plafond d’itérations permettant la convergence ; arrêt anticipé si le modèle converge avant.",
        "Méthode de choix": "Valeur standard fixée, non optimisée par recherche d’hyperparamètres.",
    },
    {
        "Modèle": "K-Means", "Hyperparamètre": "random_state",
        "Valeur choisie": hyperparams["K-Means"]["random_state"],
        "Raison": "Assurer la reproductibilité demandée.",
        "Méthode de choix": "Contrainte du projet : RANDOM_STATE = 42.",
    },
    {
        "Modèle": "DBSCAN", "Hyperparamètre": "eps",
        "Valeur choisie": round(hyperparams["DBSCAN"]["eps"], 6),
        "Raison": "Rayon de voisinage donnant le meilleur compromis mesuré parmi les candidats évaluables.",
        "Méthode de choix": "Quantiles des courbes de k-distances, puis silhouette hors bruit avec critères de couverture et de bruit.",
    },
    {
        "Modèle": "DBSCAN", "Hyperparamètre": "min_samples",
        "Valeur choisie": hyperparams["DBSCAN"]["min_samples"],
        "Raison": "Valeur retenue après comparaison explicite de 10, 22 et 42.",
        "Méthode de choix": "Comparaison des meilleurs candidats eps pour chacune des trois valeurs ; silhouette, couverture et bruit.",
    },
    {
        "Modèle": "DBSCAN", "Hyperparamètre": "metric",
        "Valeur choisie": hyperparams["DBSCAN"]["metric"],
        "Raison": "Garder la même distance pour les k-distances, la recherche de paramètres et DBSCAN.",
        "Méthode de choix": "Distance euclidienne fixée dans la méthodologie ; non optimisée.",
    },
])
for model_label in ["K-Means", "DBSCAN"]:
    print(f"Hyperparamètres — {model_label} : valeur, raison et méthode de choix")
    display(
        hyperparameter_rationale[hyperparameter_rationale["Modèle"] == model_label]
        .drop(columns="Modèle")
        .set_index("Hyperparamètre")
    )

# 3. Évaluation

Les métriques sont calculées sur le même échantillon. Pour DBSCAN, les indicateurs de qualité interne sont calculés uniquement sur les observations hors bruit ; le pourcentage de bruit et le pourcentage de points évalués sont donc affichés à côté des scores.

## 3.1 Comprendre les métriques

- **Silhouette** : mesure à quel point un individu est proche de son propre groupe et éloigné des autres. Sa plage est de **−1 à +1** ; une valeur plus élevée est préférable (les valeurs proches de 1 indiquent des groupes mieux séparés, autour de 0 des groupes qui se chevauchent et sous 0 des affectations potentiellement inadaptées).
- **Calinski-Harabasz** : compare la dispersion entre les groupes à la dispersion à l’intérieur des groupes. Il est **non négatif et sans borne supérieure fixe** ; une valeur plus élevée est généralement préférable.
- **Davies-Bouldin** : résume la similarité entre chaque groupe et son groupe le plus similaire. Il est **non négatif** ; une valeur plus faible est préférable.
- **Pourcentage de bruit** : part des observations auxquelles DBSCAN attribue `-1`, sur une plage de **0 à 100 %** ; un taux plus faible améliore la couverture, mais le bruit n’est pas automatiquement une erreur.
- **Nombre de clusters** : il n’existe pas de valeur universellement optimale. Il faut au moins deux groupes pour calculer les métriques de séparation, et un nombre de groupes suffisamment limité et interprétable pour l’objectif métier.

La recherche de paramètres utilise des sous-échantillons pour limiter le temps de calcul. Dans le tableau d’évaluation final, la silhouette est recalculée exactement sur toutes les observations évaluables (3 000 pour K-Means et les observations hors bruit pour DBSCAN). Des silhouettes proches de 0, comme celles observées dans cette analyse, signalent une séparation limitée ; il faut éviter de présenter les groupes comme une structure très nette.

In [ ]:
def evaluate_clustering(name, labels, ignore_noise=False):
    labels = np.asarray(labels)
    if ignore_noise:
        mask = labels != -1
        X_eval = X_cluster.iloc[np.flatnonzero(mask)]
        labels_eval = labels[mask]
        noise_percent = 100 * float(np.mean(labels == -1))
        clustered_labels = labels[labels != -1]
    else:
        X_eval = X_cluster
        labels_eval = labels
        noise_percent = 0.0
        clustered_labels = labels

    unique_labels = np.unique(labels_eval)
    n_clusters = len(unique_labels)
    silhouette = np.nan
    calinski = np.nan
    davies = np.nan

    if len(X_eval) > 2 and 2 <= n_clusters < len(X_eval):
        try:
            # Évaluation finale exacte sur toutes les observations évaluables.
            # La recherche des hyperparamètres reste approximée par sous-échantillonnage.
            silhouette = silhouette_score(X_eval, labels_eval)
            calinski = calinski_harabasz_score(X_eval, labels_eval)
            davies = davies_bouldin_score(X_eval, labels_eval)
        except ValueError as exc:
            print(f"Métriques non calculées pour {name} : {exc}")

    cluster_counts = pd.Series(clustered_labels).value_counts()
    largest_cluster_share = (
        100 * float(cluster_counts.max()) / len(labels)
        if len(cluster_counts) and len(labels) else np.nan
    )

    return {
        "Modèle": name,
        "Clusters (hors bruit)": n_clusters,
        "Bruit (%)": noise_percent,
        "Points évalués (%)": 100 * len(X_eval) / len(X_cluster),
        "Part du plus grand cluster (%)": largest_cluster_share,
        "Silhouette": silhouette,
        "Calinski-Harabasz": calinski,
        "Davies-Bouldin": davies,
    }

comparison = pd.DataFrame([
    evaluate_clustering("K-Means", labels_kmeans),
    evaluate_clustering("DBSCAN", labels_dbscan, ignore_noise=True),
]).set_index("Modèle")
display(comparison.round(4))

In [ ]:
# Critères de recevabilité définis avant le classement :
# 2 à 10 clusters, au moins 50 % de points évalués, au plus 50 % de bruit,
# et métriques internes calculables.
comparison_final = comparison.copy()
comparison_final["Éligible"] = (
    (comparison_final["Clusters (hors bruit)"] >= 2)
    & (comparison_final["Clusters (hors bruit)"] <= 10)
    & (comparison_final["Points évalués (%)"] >= 50)
    & (comparison_final["Bruit (%)"] <= 50)
    & comparison_final["Silhouette"].notna()
    & comparison_final["Calinski-Harabasz"].notna()
    & comparison_final["Davies-Bouldin"].notna()
)

eligible_models = comparison_final[comparison_final["Éligible"]].copy()
used_fallback_selection = False
if eligible_models.empty:
    # Repli transparent si aucun modèle ne respecte tous les seuils :
    # on classe les modèles dont les trois métriques internes restent calculables.
    eligible_models = comparison_final[
        (comparison_final["Clusters (hors bruit)"] >= 2)
        & comparison_final["Silhouette"].notna()
        & comparison_final["Calinski-Harabasz"].notna()
        & comparison_final["Davies-Bouldin"].notna()
    ].copy()
    used_fallback_selection = True
if eligible_models.empty:
    raise ValueError("Aucun modèle ne possède les métriques nécessaires au classement multicritère.")

# Classement multicritère à poids égaux :
# silhouette et Calinski-Harabasz : plus hauts préférés ;
# Davies-Bouldin, bruit et part du plus grand cluster : plus bas préférés.
# La part du plus grand cluster est utilisée ici comme indicateur d'une éventuelle
# sur-concentration, et non comme une métrique universelle de qualité du clustering.
ranking_specs = [
    ("Silhouette", False, "Rang silhouette"),
    ("Calinski-Harabasz", False, "Rang Calinski-Harabasz"),
    ("Davies-Bouldin", True, "Rang Davies-Bouldin"),
    ("Bruit (%)", True, "Rang bruit"),
    ("Part du plus grand cluster (%)", True, "Rang plus grand cluster"),
]
rank_columns = [rank_col for _, _, rank_col in ranking_specs]
for rank_col in rank_columns + ["Rang moyen multicritère"]:
    comparison_final[rank_col] = np.nan
    eligible_models[rank_col] = np.nan

for metric, ascending, rank_col in ranking_specs:
    eligible_models[rank_col] = eligible_models[metric].rank(
        ascending=ascending, method="average"
    )
    comparison_final.loc[eligible_models.index, rank_col] = eligible_models[rank_col]

eligible_models["Rang moyen multicritère"] = eligible_models[rank_columns].mean(axis=1)
comparison_final.loc[eligible_models.index, "Rang moyen multicritère"] = eligible_models["Rang moyen multicritère"]

# Le plus petit rang moyen gagne ; en cas d'égalité, silhouette puis Davies-Bouldin.
eligible_models = eligible_models.sort_values(
    ["Rang moyen multicritère", "Silhouette", "Davies-Bouldin"],
    ascending=[True, False, True],
)
best_model_name = str(eligible_models.index[0])
model_objects = {"K-Means": kmeans_model, "DBSCAN": dbscan_model}
model_labels = {"K-Means": labels_kmeans, "DBSCAN": labels_dbscan}
best_model = model_objects[best_model_name]
best_labels = np.asarray(model_labels[best_model_name]).copy()

comparison_final["Modèle retenu"] = [
    "OUI" if name == best_model_name else "Non" for name in comparison_final.index
]
comparison_final = comparison_final.rename(
    columns={"Clusters (hors bruit)": "Nombre de clusters"}
)
print(f"Modèle retenu selon le rang moyen multicritère : {best_model_name}")
print("Les cinq critères ont un poids égal ; un rang moyen plus faible est préférable.")
display(comparison_final.round(4))


## 3.2 Justification du modèle retenu

La sélection suit une règle **multicritère explicite**, après un contrôle de recevabilité : un modèle doit produire entre **2 et 10 clusters**, permettre d’évaluer au moins **50 %** de l’échantillon, laisser au plus **50 %** de bruit et fournir des valeurs calculables pour les trois indices internes.

Parmi les modèles recevables, cinq critères reçoivent un **poids égal** : (1) silhouette, à maximiser ; (2) Calinski–Harabasz, à maximiser ; (3) Davies–Bouldin, à minimiser ; (4) pourcentage de bruit, à minimiser ; et (5) part du plus grand cluster, utilisée pour repérer une éventuelle sur-concentration. Chaque critère donne un rang et le modèle retenu est celui dont le **rang moyen est le plus faible**. En cas d’égalité, la silhouette la plus élevée puis le Davies–Bouldin le plus faible servent de départage.

La part du plus grand cluster ne constitue pas une métrique universelle de qualité ; elle complète les indices internes pour juger si la segmentation produit plusieurs groupes descriptivement utilisables. La décision est donc un compromis explicite et non l’affirmation qu’un modèle domine tous les critères. Les métriques DBSCAN sont calculées hors bruit, tandis que celles de K-Means couvrent les 3 000 observations : cette différence de couverture limite la comparabilité directe des scores.

**Limites de comparaison.** L’analyse porte sur un échantillon de 3 000 personnes au maximum. Les graphiques affichent seulement la classe d’âge et l’IMC, alors que les groupes sont formés sur les 21 variables standardisées. Les silhouettes doivent être interprétées avec prudence si elles restent proches de zéro. Les résultats sont sensibles aux paramètres et restent descriptifs : ils ne constituent ni un diagnostic individuel ni une preuve de causalité.

La distance euclidienne traite les variables binaires et ordinales comme des valeurs numériques et peut donc imposer une géométrie discutable à ces catégories. Une perspective méthodologique serait d’étudier, dans un travail ultérieur, une distance de Gower ou K-Prototypes ; ces approches ne sont pas appliquées ici.


In [ ]:
from IPython.display import Markdown

def format_metric(value, digits=4):
    return "non calculable" if pd.isna(value) else f"{value:.{digits}f}"

best_row = comparison_final.loc[best_model_name]
rank_score = best_row["Rang moyen multicritère"]
lines = [
    f"**Modèle retenu selon le classement multicritère : {best_model_name}.**",
    f"Son rang moyen est de {format_metric(rank_score, 2)} ; le rang le plus faible est préférable.",
    "Les cinq critères sont pondérés de manière égale : silhouette, Calinski-Harabasz, Davies-Bouldin, bruit et part du plus grand cluster.",
    "",
]
for model_label, row in comparison_final.iterrows():
    lines.append(
        f"- **{model_label}** — clusters : {int(row['Nombre de clusters'])} ; "
        f"silhouette exacte : {format_metric(row['Silhouette'])} ; "
        f"Calinski-Harabasz : {format_metric(row['Calinski-Harabasz'], 1)} ; "
        f"Davies-Bouldin : {format_metric(row['Davies-Bouldin'])} ; "
        f"bruit : {format_metric(row['Bruit (%)'], 2)} % ; "
        f"points évalués : {format_metric(row['Points évalués (%)'], 2)} % ; "
        f"plus grand cluster : {format_metric(row['Part du plus grand cluster (%)'], 2)} % ; "
        f"rang moyen : {format_metric(row['Rang moyen multicritère'], 2)} ; "
        f"recevable : {'oui' if row['Éligible'] else 'non'} ; "
        f"retenu : {'oui' if model_label == best_model_name else 'non'}."
    )

if used_fallback_selection:
    lines += ["", "**Prudence :** aucun modèle ne satisfait tous les seuils de recevabilité ; le classement est appliqué aux modèles disposant des trois métriques internes calculables."]

# Présenter les éventuels contre-indices plutôt que de les masquer.
if "K-Means" in comparison_final.index and "DBSCAN" in comparison_final.index:
    km_row = comparison_final.loc["K-Means"]
    db_row = comparison_final.loc["DBSCAN"]
    if pd.notna(km_row["Silhouette"]) and pd.notna(db_row["Silhouette"]):
        favored = "K-Means" if km_row["Silhouette"] > db_row["Silhouette"] else ("DBSCAN" if db_row["Silhouette"] > km_row["Silhouette"] else "les deux modèles à égalité")
        lines.append(f"La silhouette favorise {favored}.")
    if pd.notna(km_row["Calinski-Harabasz"]) and pd.notna(db_row["Calinski-Harabasz"]):
        favored = "K-Means" if km_row["Calinski-Harabasz"] > db_row["Calinski-Harabasz"] else ("DBSCAN" if db_row["Calinski-Harabasz"] > km_row["Calinski-Harabasz"] else "les deux modèles à égalité")
        lines.append(f"L’indice de Calinski-Harabasz favorise {favored}.")
    if pd.notna(db_row["Part du plus grand cluster (%)"]) and db_row["Part du plus grand cluster (%)"] > 90:
        lines.append(
            f"**Limite de segmentation :** le plus grand cluster DBSCAN représente {db_row['Part du plus grand cluster (%)']:.2f} % de tout l’échantillon. Cette concentration peut limiter l’intérêt de DBSCAN pour distinguer plusieurs profils de prévention, même si certaines métriques internes sont meilleures."
        )
    if pd.notna(km_row["Part du plus grand cluster (%)"]) and pd.notna(db_row["Part du plus grand cluster (%)"]) and km_row["Part du plus grand cluster (%)"] < db_row["Part du plus grand cluster (%)"]:
        lines.append(
            f"K-Means est moins concentré dans un seul groupe ({km_row['Part du plus grand cluster (%)']:.2f} % dans son plus grand cluster contre {db_row['Part du plus grand cluster (%)']:.2f} % pour DBSCAN). Ce résultat entre dans le classement comme un indicateur de répartition, sans garantir à lui seul une meilleure segmentation."
        )

display(Markdown("\n\n".join(lines)))


In [ ]:
# Effectif et pourcentage d'individus dans chaque cluster ou dans le bruit.
cluster_size_rows = []
for model_name, labels in [
    ("K-Means", labels_kmeans),
    ("DBSCAN", labels_dbscan),
]:
    counts = pd.Series(labels).value_counts().sort_index()
    for cluster_id, count in counts.items():
        cluster_label = "Bruit (-1)" if model_name == "DBSCAN" and int(cluster_id) == -1 else f"Cluster {int(cluster_id)}"
        cluster_size_rows.append({
            "Modèle": model_name,
            "Cluster": int(cluster_id),
            "Nom du groupe": cluster_label,
            "Effectif": int(count),
            "Pourcentage": round(100 * count / len(labels), 2),
        })
cluster_sizes_comparison = pd.DataFrame(cluster_size_rows)
display(cluster_sizes_comparison)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, model_name in zip(axes, ["K-Means", "DBSCAN"]):
    data = cluster_sizes_comparison[cluster_sizes_comparison["Modèle"] == model_name].copy()
    data = data.sort_values("Effectif", ascending=False).head(15)
    labels_for_plot = data["Nom du groupe"].tolist()
    ax.bar(labels_for_plot, data["Pourcentage"])
    ax.set_title(f"{model_name} — groupes les plus nombreux")
    ax.set_xlabel("Groupe")
    ax.set_ylabel("Individus (%)")
    ax.tick_params(axis="x", rotation=60)
plt.tight_layout()
plt.show()

**Interprétation.** La comparaison des tailles met en évidence la répartition des effectifs entre groupes et aide à repérer un groupe dominant. Pour DBSCAN, la catégorie « Bruit (−1) » est distinguée des clusters ordinaires.

## 3.3 Profils des groupes

Les profils sont décrits avec les moyennes des variables d’origine. Pour une variable binaire codée 0/1, la moyenne correspond à la proportion d’individus ayant la valeur 1.

In [ ]:
df_results = df_cluster_source.copy()
for col in selected_features:
    df_results[col] = X.loc[X_cluster.index, col].to_numpy()
df_results["Cluster_KMeans"] = labels_kmeans
df_results["Cluster_DBSCAN"] = labels_dbscan

profile_cols = [col for col in selected_features if col in df_results.columns]
profile_kmeans = df_results.groupby("Cluster_KMeans")[profile_cols].mean()
profile_dbscan = df_results.groupby("Cluster_DBSCAN")[profile_cols].mean()

print("Profils moyens — K-Means")
display(profile_kmeans.round(2))
print("Profils moyens — DBSCAN (−1 = bruit)")
display(profile_dbscan.round(2))


**Interprétation.** Une moyenne de 0,75 pour une variable binaire codée 0/1 correspond à environ 75 % de valeurs égales à 1 dans le groupe. Pour une variable ordinale, la moyenne résume les codes et ne doit pas être interprétée comme une mesure directe.

In [ ]:
profile_features_to_plot = [
    col for col in ["HighBP", "HighChol", "BMI", "Smoker", "HeartDiseaseorAttack",
                    "PhysActivity", "GenHlth", "DiffWalk"]
    if col in profile_cols
]
# Les barres ne comparent que des proportions (0 à 1) : BMI et GenHlth, à une autre échelle, sont traités dans la carte thermique.
profile_binary_to_plot = [col for col in profile_features_to_plot if col not in ("BMI", "GenHlth")]

if profile_binary_to_plot:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
    profile_kmeans[profile_binary_to_plot].plot(kind="bar", ax=axes[0])
    axes[0].set_title("Proportions par groupe — K-Means")

    dbscan_non_noise_ids = [idx for idx in profile_dbscan.index if idx != -1]
    if dbscan_non_noise_ids:
        dbscan_order = (df_results.loc[df_results["Cluster_DBSCAN"] != -1, "Cluster_DBSCAN"]
                        .value_counts().head(10).index.tolist())
        profile_dbscan.loc[dbscan_order, profile_binary_to_plot].plot(kind="bar", ax=axes[1])
        axes[1].set_title("Proportions par groupe — DBSCAN (hors bruit, 10 plus grands clusters)")
    else:
        axes[1].text(0.5, 0.5, "Aucun cluster hors bruit", ha="center", va="center")
        axes[1].set_title("Proportions par groupe — DBSCAN")
    for ax in axes:
        ax.set_xlabel("Cluster")
        ax.set_ylabel("Proportion d’individus (0 à 1)")
        ax.tick_params(axis="x", rotation=0)
        ax.legend(fontsize=8, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.15))
    plt.tight_layout()
    plt.show()
else:
    print("Aucune des variables choisies pour ce graphique n’est disponible.")


**Interprétation.** Chaque barre indique la proportion d’individus d’un groupe présentant la caractéristique (par exemple hypertension, cholestérol élevé, tabagisme, maladie cardiaque, activité physique ou difficulté à marcher). Des proportions simultanément élevées décrivent un profil qui peut mériter une attention descriptive dans une démarche de prévention ; cela ne prouve ni causalité ni efficacité d’une intervention.

In [ ]:
if profile_features_to_plot:
    # Couleur : écart à la moyenne de l’échantillon, en écarts-types (échelles comparables, IMC inclus).
    # Valeur affichée : la moyenne réelle du groupe.
    sample_mean = df_results[profile_features_to_plot].mean()
    sample_std = df_results[profile_features_to_plot].std().replace(0, np.nan)

    def deviation_table(profile):
        return ((profile[profile_features_to_plot] - sample_mean) / sample_std).T

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    sns.heatmap(deviation_table(profile_kmeans), annot=profile_kmeans[profile_features_to_plot].round(2).T,
                fmt="", cmap="coolwarm", center=0, vmin=-1.5, vmax=1.5, ax=axes[0],
                cbar_kws={"label": "Écart à la moyenne (écarts-types)"})
    axes[0].set_title("Profils — K-Means")

    dbscan_non_noise_ids = [idx for idx in profile_dbscan.index if idx != -1]
    if dbscan_non_noise_ids:
        profile_dbscan_clusters = profile_dbscan.loc[dbscan_non_noise_ids]
        sns.heatmap(deviation_table(profile_dbscan_clusters),
                    annot=profile_dbscan_clusters[profile_features_to_plot].round(2).T,
                    fmt="", cmap="coolwarm", center=0, vmin=-1.5, vmax=1.5, ax=axes[1],
                    cbar_kws={"label": "Écart à la moyenne (écarts-types)"})
    else:
        axes[1].text(0.5, 0.5, "Aucun cluster hors bruit", ha="center", va="center")
    axes[1].set_title("Profils — DBSCAN (hors bruit)")
    for ax in axes:
        ax.set_xlabel("Cluster")
        ax.set_ylabel("Variable")
    plt.tight_layout()
    plt.show()


**Interprétation.** Chaque case indique la moyenne réelle du groupe ; la couleur indique son écart à la moyenne de l’échantillon, en écarts-types. Les couleurs mettent en évidence des caractéristiques distinctives, mais ne constituent ni une estimation de risque individuel ni une preuve de causalité.

In [ ]:
# Constats calculés à partir des profils observés, sans utiliser la cible pour former les groupes.
from IPython.display import Markdown

findings = [
    "### Constats descriptifs sur les groupes",
    "Les points ci-dessous sont calculés à partir des effectifs et moyennes du présent échantillon. Les moyennes des variables binaires codées 0/1 sont interprétées comme des proportions."
]

# Repérer automatiquement les groupes où aucun individu ne déclare avoir eu un contrôle du cholestérol.
if "CholCheck" in profile_kmeans.columns:
    for cluster_id, row in profile_kmeans.iterrows():
        if float(row["CholCheck"]) <= 1e-12:
            mask = df_results["Cluster_KMeans"] == cluster_id
            count = int(mask.sum())
            share = 100 * count / len(df_results)
            findings.append(
                f"- **K-Means, cluster {int(cluster_id)} :** {count} personnes ({share:.2f} % de l’échantillon) ont `CholCheck = 0` dans ce groupe. Il s’agit d’un profil caractérisé par l’absence de contrôle du cholestérol déclaré, et non d’un indicateur de bonne santé."
            )

if "CholCheck" in profile_dbscan.columns:
    for cluster_id, row in profile_dbscan.iterrows():
        if int(cluster_id) == -1:
            continue
        if float(row["CholCheck"]) <= 1e-12:
            mask = df_results["Cluster_DBSCAN"] == cluster_id
            count = int(mask.sum())
            share = 100 * count / len(df_results)
            extra = ""
            if target_col and target_col in df_results:
                target_values = pd.to_numeric(df_results.loc[mask, target_col], errors="coerce").dropna()
                if len(target_values) and set(target_values.unique()).issubset({0, 1}):
                    rate = 100 * float(target_values.mean())
                    extra = f" Le statut diabétique observé est de {rate:.2f} % dans ce groupe ; cela ne doit pas être interprété comme un effet protecteur."
            findings.append(
                f"- **DBSCAN, cluster {int(cluster_id)} :** {count} personnes ({share:.2f} % de l’échantillon) ont `CholCheck = 0`. Ce groupe reflète d’abord une caractéristique de dépistage déclaré.{extra}"
            )

# Rechercher les groupes entièrement définis par la présence déclarée d'un antécédent d'AVC.
if "Stroke" in profile_dbscan.columns:
    for cluster_id, row in profile_dbscan.iterrows():
        if int(cluster_id) == -1:
            continue
        if float(row["Stroke"]) >= 1 - 1e-12:
            mask = df_results["Cluster_DBSCAN"] == cluster_id
            count = int(mask.sum())
            share = 100 * count / len(df_results)
            findings.append(
                f"- **DBSCAN, cluster {int(cluster_id)} :** les {count} personnes ({share:.2f} % de l’échantillon) ont `Stroke = 1` dans les données. Ce groupe est donc fortement marqué par cette variable ; sa faible taille et le rôle des variables binaires doivent être pris en compte."
            )

# Identifier dynamiquement le cluster K-Means avec la proportion d'hypertension la plus élevée.
if "HighBP" in profile_kmeans.columns:
    highbp_cluster = profile_kmeans["HighBP"].idxmax()
    mask = df_results["Cluster_KMeans"] == highbp_cluster
    count = int(mask.sum())
    share = 100 * count / len(df_results)
    highbp_rate = 100 * float(profile_kmeans.loc[highbp_cluster, "HighBP"])
    diabetes_note = ""
    if target_col and target_col in df_results:
        target_values = pd.to_numeric(df_results.loc[mask, target_col], errors="coerce").dropna()
        if len(target_values) and set(target_values.unique()).issubset({0, 1}):
            diabetes_note = f" Le statut diabétique observé dans ce groupe est de {100 * float(target_values.mean()):.2f} %."
    findings.append(
        f"- **Groupe K-Means le plus marqué par `HighBP` :** cluster {int(highbp_cluster)}, {count} personnes ({share:.2f} %), dont {highbp_rate:.2f} % ont `HighBP = 1` dans les données.{diabetes_note} Cette concentration descriptive peut être plus utile pour discuter plusieurs profils que la seule qualité métrique globale, sans inférer de causalité."
    )

findings.append(
    "**Prudence d’interprétation.** Les clusters peuvent être structurés par une variable binaire rare après standardisation. Les petits groupes définis presque entièrement par `CholCheck` ou `Stroke` doivent donc être lus comme des artefacts possibles de la géométrie des distances, et non automatiquement comme des profils complets de santé."
)
display(Markdown("\n\n".join(findings)))


## 3.4 Description du statut diabétique par groupe

Cette étape compare la répartition du statut diabétique aux groupes obtenus, après l’apprentissage non supervisé.

In [ ]:
if target_col:
    for model_name, cluster_col in [
        ("K-Means", "Cluster_KMeans"),
        ("DBSCAN", "Cluster_DBSCAN"),
    ]:
        distribution = pd.crosstab(
            df_results[cluster_col], df_results[target_col], normalize="index"
        ).mul(100)
        print(f"Statut diabétique par groupe — {model_name} (%)")
        display(distribution.round(2))

        ax = distribution.plot(kind="bar", stacked=True, figsize=(8, 4), colormap="tab20")
        ax.set_title(f"Répartition du statut diabétique — {model_name}")
        ax.set_xlabel("Groupe" + (" (−1 = bruit)" if model_name == "DBSCAN" else ""))
        ax.set_ylabel("Pourcentage (%)")
        ax.legend(title=target_col, bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.tight_layout()
        plt.show()
else:
    print("Aucune colonne de statut diabétique trouvée. L’analyse descriptive de la cible est ignorée.")


**Interprétation.** Le statut diabétique est décrit après le clustering et n’a pas servi à former les groupes. Les répartitions révèlent d’éventuelles associations, pas une relation de cause à effet ; chaque catégorie est affichée si la cible en comprend plusieurs.

# 4. Déploiement

Cette section rend le modèle retenu réutilisable sur de nouvelles observations. L’artefact sauvegardé contient uniquement le modèle sélectionné, le `StandardScaler` ajusté, la liste ordonnée des variables et les valeurs d’imputation utilisées lors de la préparation des données. Lorsque DBSCAN est retenu, les observations d’entraînement classées hors bruit (points noyaux et points frontière) et leurs étiquettes sont également conservées pour l’affectation approchée des nouvelles observations.


## 4.1 Fonctions de chargement et de prédiction

La fonction `load_artifact()` recharge et vérifie le fichier sauvegardé. La fonction `predict_profile(new_data, artifact)` applique les mêmes conversions, imputations et standardisation qu’à l’entraînement, dans le même ordre de variables. Les colonnes supplémentaires sont ignorées, mais toutes les variables requises doivent être présentes.

Le comportement dépend du modèle retenu : K-Means affecte chaque observation au centroïde le plus proche. DBSCAN ne fournit pas de méthode `predict` native ; une règle d’affectation approchée recherche l’observation d’entraînement non bruitée la plus proche et utilise `eps` comme seuil.


In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors


def load_artifact(path="model_cdc_diabetes.joblib"):
    """Charge et vérifie l’artefact de déploiement sauvegardé avec joblib."""
    artifact_path = Path(path)
    if not artifact_path.is_file():
        raise FileNotFoundError(
            f"Artefact introuvable : {artifact_path.resolve()}. "
            "Exécutez d’abord la cellule de sauvegarde du modèle."
        )

    artifact = joblib.load(artifact_path)
    required_keys = {
        "scaler", "model", "best_model_name", "selected_features", "fill_values"
    }
    missing_keys = required_keys.difference(artifact)
    if missing_keys:
        raise ValueError(
            "Artefact incomplet ; clés manquantes : " + ", ".join(sorted(missing_keys))
        )
    if artifact["best_model_name"] == "DBSCAN":
        dbscan_keys = {"dbscan_reference_points", "dbscan_reference_labels", "dbscan_eps", "dbscan_metric"}
        missing_dbscan_keys = dbscan_keys.difference(artifact)
        if missing_dbscan_keys:
            raise ValueError(
                "Artefact DBSCAN incomplet ; clés manquantes : "
                + ", ".join(sorted(missing_dbscan_keys))
            )
    return artifact


def predict_profile(new_data: pd.DataFrame, artifact: dict) -> pd.DataFrame:
    """Prétraite de nouvelles observations et prédit leur groupe.

    Les valeurs manquantes ou non numériques sont remplacées par les valeurs
    d’imputation mémorisées lors de la préparation de l’échantillon d’entraînement.
    Le scaler n’est jamais réajusté sur les nouvelles observations.

    Pour DBSCAN, l’affectation est approchée : le label de l’observation d’entraînement
    non bruitée la plus proche est retourné si sa distance euclidienne standardisée est
    <= eps ; sinon, le label -1 est attribué.
    """
    if not isinstance(new_data, pd.DataFrame):
        raise TypeError("new_data doit être un pandas.DataFrame.")
    if new_data.empty:
        raise ValueError("new_data ne doit pas être vide.")

    features = list(artifact["selected_features"])
    missing_features = [col for col in features if col not in new_data.columns]
    if missing_features:
        raise ValueError(
            "Variables requises absentes : " + ", ".join(missing_features)
        )

    data = new_data.loc[:, features].copy()
    data = data.apply(pd.to_numeric, errors="coerce")
    data = data.replace([np.inf, -np.inf], np.nan)

    fill_values = artifact["fill_values"]
    for col in features:
        data[col] = data[col].fillna(fill_values[col])

    X_new_scaled = pd.DataFrame(
        artifact["scaler"].transform(data[features]),
        columns=features,
        index=new_data.index,
    )

    model_name = artifact["best_model_name"]
    if model_name == "K-Means":
        predicted_labels = artifact["model"].predict(X_new_scaled)
        return pd.DataFrame(
            {"Groupe prédit": predicted_labels.astype(int)}, index=new_data.index
        )

    if model_name == "DBSCAN":
        reference_points = np.asarray(artifact.get("dbscan_reference_points", []), dtype=float)
        reference_labels = np.asarray(artifact.get("dbscan_reference_labels", []), dtype=int)
        eps = float(artifact["dbscan_eps"])
        metric = artifact.get("dbscan_metric", "euclidean")
        if metric != "euclidean":
            raise ValueError(
                "La prédiction DBSCAN de ce notebook prend en charge la distance euclidienne uniquement."
            )

        if reference_points.size == 0:
            predicted_labels = np.full(len(data), -1, dtype=int)
            nearest_distances = np.full(len(data), np.nan, dtype=float)
        else:
            if reference_points.ndim != 2 or reference_points.shape[1] != len(features):
                raise ValueError("Les observations de référence DBSCAN ont une forme incompatible avec les variables attendues.")
            if len(reference_points) != len(reference_labels):
                raise ValueError("Les observations de référence et leurs étiquettes DBSCAN ne sont pas alignées.")
            if np.any(reference_labels == -1):
                raise ValueError("Les références DBSCAN doivent exclure les observations étiquetées comme bruit (-1).")
            nearest = NearestNeighbors(n_neighbors=1, metric=metric)
            nearest.fit(reference_points)
            distances, indices = nearest.kneighbors(X_new_scaled.to_numpy())
            nearest_distances = distances[:, 0]
            nearest_labels = reference_labels[indices[:, 0]]
            predicted_labels = np.where(nearest_distances <= eps, nearest_labels, -1)

        return pd.DataFrame(
            {
                "Groupe prédit": predicted_labels.astype(int),
                "Distance au point non bruité le plus proche": nearest_distances,
            },
            index=new_data.index,
        )

    raise ValueError(f"Modèle non pris en charge dans l’artefact : {model_name!r}")


## 4.2 Construction et sauvegarde de l’artefact

L’artefact est assemblé à partir des objets déjà ajustés dans les sections précédentes. Il conserve uniquement le modèle retenu par la règle de sélection de la section 3, et non les deux modèles candidats. Le scaler et les valeurs d’imputation correspondent exactement au prétraitement utilisé pour l’apprentissage.

Si DBSCAN est retenu, les observations d’entraînement classées hors bruit et leurs étiquettes sont enregistrées. La règle d’affectation utilise ce référentiel non bruité, qui comprend les points noyaux et les points frontière. Si K-Means est retenu, ces informations spécifiques à DBSCAN ne sont pas ajoutées à l’artefact.


In [ ]:
deployment_artifact = {
    "scaler": scaler,
    "model": best_model,
    "best_model_name": best_model_name,
    "selected_features": list(selected_features),
    "fill_values": {col: float(fill_values[col]) for col in selected_features},
    "random_state": int(RANDOM_STATE),
    "selected_hyperparams": dict(hyperparams[best_model_name]),
}

if best_model_name == "DBSCAN":
    non_noise_mask = labels_dbscan != -1
    deployment_artifact["dbscan_reference_points"] = X_cluster.to_numpy()[non_noise_mask]
    deployment_artifact["dbscan_reference_labels"] = np.asarray(
        labels_dbscan[non_noise_mask], dtype=int
    )
    deployment_artifact["dbscan_eps"] = float(dbscan_eps)
    deployment_artifact["dbscan_metric"] = str(hyperparams["DBSCAN"]["metric"])

artifact_path = "model_cdc_diabetes.joblib"
joblib.dump(deployment_artifact, artifact_path)

# Recharger le fichier pour vérifier que l’artefact sauvegardé est lisible.
deployment_artifact = load_artifact(artifact_path)
print(f"Artefact sauvegardé et rechargé : {artifact_path}")
print(f"Modèle inclus dans l’artefact : {deployment_artifact['best_model_name']}")
print(f"Nombre de variables attendues : {len(deployment_artifact['selected_features'])}")
print("Modèles inclus : 1 (le modèle retenu par l’évaluation).")


## 4.3 Vérification fonctionnelle du déploiement

La première vérification reprend cinq lignes brutes dans le même ordre que l’échantillon et compare leurs prédictions à `best_labels[:5]`. Elle teste la cohérence du prétraitement, mais à elle seule, elle peut ne couvrir qu’un seul groupe.

Une seconde vérification choisit des observations représentatives de chaque étiquette disponible, y compris `-1` pour DBSCAN. Pour K-Means, les étiquettes de référence doivent être reproduites. Pour DBSCAN, les étiquettes sont comparées à titre diagnostique : une affectation de nouveaux points fondée sur le voisin non bruité le plus proche est une approximation et ne reproduit pas nécessairement la décision d’origine pour les points frontière ou bruit.

Enfin, cinq observations provenant du jeu source mais non utilisées dans l’échantillon de clustering sont soumises à la fonction. Comme elles n’ont pas d’étiquette de référence calculée par le modèle entraîné, ce test vérifie que le déploiement peut traiter de nouvelles lignes sans réentraînement.

In [ ]:
# Test simple sur cinq lignes brutes : contrôle de compatibilité avec best_labels[:5].
test_rows = df.loc[X_cluster.index[:5], selected_features].copy()
test_predictions = predict_profile(test_rows, deployment_artifact)
expected_labels = np.asarray(best_labels[:len(test_rows)], dtype=int)
predicted_labels = test_predictions["Groupe prédit"].to_numpy(dtype=int)
row_comparison = pd.DataFrame({
    "Étiquette d’entraînement": expected_labels,
    "Étiquette prédite au déploiement": predicted_labels,
    "Concordance": predicted_labels == expected_labels,
})
print("Prédictions sur cinq observations brutes :")
display(pd.concat(
    [test_rows.reset_index(drop=True), test_predictions.reset_index(drop=True)], axis=1
))
print("Comparaison avec les étiquettes de référence :")
display(row_comparison)
predictions_match = bool(np.array_equal(predicted_labels, expected_labels))
print(f"Prédictions identiques à best_labels[:5] : {predictions_match}")
if best_model_name == "K-Means" and not predictions_match:
    raise AssertionError(
        "Les prédictions K-Means ne correspondent pas aux étiquettes d’entraînement ; "
        "vérifiez l’ordre des variables et le prétraitement."
    )
if best_model_name == "DBSCAN" and not predictions_match:
    print(
        "Note : le test initial peut diverger avec DBSCAN, car le bruit et les points frontière "
        "sont réaffectés par une règle approchée. Le test de couverture ci-dessous détaille ce comportement."
    )

# Test de couverture : au moins une observation représentative pour chaque étiquette d'entraînement.
representative_positions = []
for label in sorted(np.unique(best_labels)):
    positions = np.flatnonzero(best_labels == label)
    if len(positions) == 0:
        continue
    if best_model_name == "DBSCAN" and label == -1:
        # Choisir, parmi les points d'origine classés bruit, celui le plus éloigné des références non bruitées.
        ref_points = np.asarray(deployment_artifact.get("dbscan_reference_points", []), dtype=float)
        if ref_points.size:
            nearest_reference = NearestNeighbors(n_neighbors=1, metric="euclidean").fit(ref_points)
            distances, _ = nearest_reference.kneighbors(X_cluster.iloc[positions].to_numpy())
            chosen_position = int(positions[int(np.argmax(distances[:, 0]))])
        else:
            chosen_position = int(positions[0])
    else:
        # Une observation réelle du groupe ; sa distance à elle-même est nulle dans le référentiel.
        chosen_position = int(positions[0])
    representative_positions.append(chosen_position)

coverage_rows = df.loc[X_cluster.index[representative_positions], selected_features].copy()
coverage_expected = np.asarray(best_labels[representative_positions], dtype=int)
coverage_predicted_df = predict_profile(coverage_rows, deployment_artifact)
coverage_predicted = coverage_predicted_df["Groupe prédit"].to_numpy(dtype=int)
coverage_report = pd.DataFrame({
    "Étiquette d’entraînement": coverage_expected,
    "Étiquette prédite": coverage_predicted,
    "Concordance": coverage_expected == coverage_predicted,
}, index=coverage_rows.index)
print("Test de couverture des étiquettes disponibles (dont le bruit -1, si présent) :")
display(coverage_report)
if best_model_name == "K-Means" and not coverage_report["Concordance"].all():
    raise AssertionError("Le test de couverture K-Means n’a pas reproduit toutes les étiquettes de référence.")
if best_model_name == "DBSCAN" and not coverage_report["Concordance"].all():
    print(
        "Au moins une étiquette DBSCAN diffère : ce résultat illustre les limites de la règle d’affectation approchée. "
        "Inspecter la distance au point de référence le plus proche et ne pas interpréter cette règle comme un DBSCAN natif transformé."
    )

# Test hors échantillon : données sources qui n'ont pas servi au clustering.
held_out_indices = df.index[~df.index.isin(X_cluster.index)][:5]
held_out_rows = df.loc[held_out_indices, selected_features].copy()
held_out_predictions = predict_profile(held_out_rows, deployment_artifact)
print("Prédictions sur cinq observations de la source non utilisées pour ajuster le clustering :")
display(pd.concat(
    [held_out_rows.reset_index(drop=True), held_out_predictions.reset_index(drop=True)], axis=1
))
assert len(held_out_predictions) == len(held_out_rows) == 5
print("Test hors échantillon réussi : cinq nouvelles lignes ont été traitées sans réentraînement.")


## 4.4 Réutilisation de l’artefact dans une autre session Jupyter

Le fichier `model_cdc_diabetes.joblib` contient le modèle retenu et les objets de prétraitement nécessaires. Pour utiliser l’artefact sans réentraîner les modèles, charger le fichier avec `load_artifact()`, puis appeler `predict_profile()` sur un DataFrame contenant les variables requises dans leur format d’origine. Les fonctions et leurs dépendances sont définies dans ce notebook ; aucun fichier `deploy_cdc.py` séparé n’est nécessaire.

Les variables attendues sont accessibles dans `artifact["selected_features"]`. Les valeurs manquantes à l’intérieur des colonnes présentes sont imputées avec les valeurs mémorisées lors de la préparation des données. Les colonnes requises absentes déclenchent une erreur explicite.


In [ ]:
# Exemple autonome à exécuter après la définition des deux fonctions ci-dessus.
from IPython.display import display

artifact = load_artifact("model_cdc_diabetes.joblib")

exemple = {
    "HighBP": 0, "HighChol": 0, "CholCheck": 1, "BMI": 27,
    "Smoker": 0, "Stroke": 0, "HeartDiseaseorAttack": 0,
    "PhysActivity": 1, "Fruits": 1, "Veggies": 1,
    "HvyAlcoholConsump": 0, "AnyHealthcare": 1, "NoDocbcCost": 0,
    "GenHlth": 2, "MentHlth": 0, "PhysHlth": 0, "DiffWalk": 0,
    "Sex": 1, "Age": 9, "Education": 5, "Income": 6,
}
new_data = pd.DataFrame([exemple], columns=artifact["selected_features"])
resultats = predict_profile(new_data, artifact)
display(resultats)

In [ ]:
from IPython.display import Markdown

concl_row = comparison_final.loc[best_model_name]
cluster_n = int(concl_row["Nombre de clusters"])
rank_score = concl_row["Rang moyen multicritère"]
dbscan_share = (
    float(comparison_final.loc["DBSCAN", "Part du plus grand cluster (%)"])
    if "DBSCAN" in comparison_final.index and pd.notna(comparison_final.loc["DBSCAN", "Part du plus grand cluster (%)"])
    else np.nan
)
km_share = (
    float(comparison_final.loc["K-Means", "Part du plus grand cluster (%)"])
    if "K-Means" in comparison_final.index and pd.notna(comparison_final.loc["K-Means", "Part du plus grand cluster (%)"])
    else np.nan
)
selection_sentence = (
    f"**Modèle retenu selon le classement multicritère :** {best_model_name}, rang moyen {format_metric(rank_score, 2)}, "
    f"silhouette {format_metric(concl_row['Silhouette'])}, Calinski-Harabasz {format_metric(concl_row['Calinski-Harabasz'], 1)}, "
    f"Davies-Bouldin {format_metric(concl_row['Davies-Bouldin'])}, {cluster_n} groupes et bruit {format_metric(concl_row['Bruit (%)'], 1)} %."
)
if pd.notna(dbscan_share) and dbscan_share > 90:
    selection_sentence += (
        f" Le plus grand groupe DBSCAN représente toutefois {dbscan_share:.2f} % de l’échantillon ; ce déséquilibre est pris en compte dans le classement multicritère."
    )
    if pd.notna(km_share) and km_share < dbscan_share:
        selection_sentence += f" K-Means est moins concentré dans un seul groupe ({km_share:.2f} %)."

# Fait descriptif calculé pour l'enseignement principal, sans interprétation causale.
insight = "Les groupes décrivent des combinaisons de caractéristiques, sans établir de relation causale."
if "HighBP" in profile_kmeans.columns:
    cluster_id = profile_kmeans["HighBP"].idxmax()
    cluster_mask = df_results["Cluster_KMeans"] == cluster_id
    share = 100 * float(cluster_mask.sum()) / len(df_results)
    bp_rate = 100 * float(profile_kmeans.loc[cluster_id, "HighBP"])
    insight = f"Le cluster K-Means {int(cluster_id)} regroupe {share:.1f} % de l’échantillon et {bp_rate:.1f} % de ses membres ont une hypertension déclarée ; cette observation est descriptive, non causale."

conclusion_lines = [
    "## Conclusion",
    "- **BO atteint ?** Partiellement : des profils exploratoires sont décrits, mais leur utilité pour la prévention doit être validée par des experts métier.",
    f"- {selection_sentence}",
    f"- **Enseignement principal :** {insight} Le statut diabétique a été utilisé uniquement pour décrire les groupes après l’apprentissage.",
    "- **Limites et perspectives :** échantillon de 3 000 personnes, métriques DBSCAN hors bruit, visualisation sur deux variables, sensibilité aux paramètres et distance euclidienne sur variables binaires/ordinales ; étudier la stabilité, Gower ou K-Prototypes dans des travaux futurs."
]
display(Markdown("\n\n".join(conclusion_lines)))


## 4.6 Export des statistiques pour l’application web

Cette cellule exporte les statistiques agrégées du modèle retenu vers `cluster_dashboard_data.json`. Elle ne réentraîne aucun modèle. Les valeurs proviennent des groupes calculés par le notebook et le statut diabétique est décrit après le clustering, sans participer à l’apprentissage. Le fichier `model_cdc_diabetes.joblib` reste l’artefact de prédiction.


In [ ]:
import json
import platform
import sklearn


def _json_number(value):
    """Convertit un nombre Pandas/NumPy en valeur JSON finie ou None."""
    if value is None or pd.isna(value):
        return None
    value = float(value)
    return value if np.isfinite(value) else None


selected_cluster_column = "Cluster_KMeans" if best_model_name == "K-Means" else "Cluster_DBSCAN"
selected_counts = pd.Series(best_labels, name="cluster_id").value_counts().sort_index()
cluster_sizes_payload = []
for cluster_id, count in selected_counts.items():
    cluster_id = int(cluster_id)
    is_noise = best_model_name == "DBSCAN" and cluster_id == -1
    cluster_sizes_payload.append({
        "cluster_id": cluster_id,
        "label": "Bruit (-1)" if is_noise else f"Cluster {cluster_id}",
        "count": int(count),
        "percentage": _json_number(100 * count / len(best_labels)),
        "is_noise": bool(is_noise),
    })

selected_profiles = df_results.groupby(selected_cluster_column)[selected_features].mean(numeric_only=True)
cluster_profiles_payload = []
for cluster_id, row in selected_profiles.iterrows():
    cluster_id = int(cluster_id)
    profile = {
        "cluster_id": cluster_id,
        "label": "Bruit (-1)" if best_model_name == "DBSCAN" and cluster_id == -1 else f"Cluster {cluster_id}",
        "is_noise": bool(best_model_name == "DBSCAN" and cluster_id == -1),
        "values": {feature: _json_number(row[feature]) for feature in selected_features},
    }
    # Le statut est ajouté uniquement pour décrire a posteriori la composition des groupes.
    if target_col and target_col in df_results.columns:
        group_target = df_results.loc[df_results[selected_cluster_column] == cluster_id, target_col]
        distribution = group_target.value_counts(normalize=True, dropna=False).sort_index()
        profile["target_distribution_pct"] = {str(k): _json_number(v * 100) for k, v in distribution.items()}
        profile["diabetes_prevalence_pct"] = _json_number(100 * float((group_target == 2).mean())) if target_col == "Diabetes_012" else None
    cluster_profiles_payload.append(profile)

metric_row = comparison_final.loc[best_model_name]
metrics_payload = {
    "silhouette": _json_number(metric_row.get("Silhouette")),
    "calinski_harabasz": _json_number(metric_row.get("Calinski-Harabasz")),
    "davies_bouldin": _json_number(metric_row.get("Davies-Bouldin")),
    "noise_pct": _json_number(metric_row.get("Bruit (%)")),
    "n_clusters": int(metric_row.get("Nombre de clusters", 0)),
    "largest_cluster_pct": _json_number(metric_row.get("Part du plus grand cluster (%)")),
    "evaluated_pct": _json_number(metric_row.get("Points évalués (%)")),
}

overall_means = df_results[selected_features].mean(numeric_only=True)
overall_stds = df_results[selected_features].std(numeric_only=True, ddof=0).replace(0, np.nan)
plot_frame = visualization_data[["Age_jitter", "BMI"]].copy()
plot_frame["cluster_id"] = best_labels
plot_frame = plot_frame.sample(n=min(900, len(plot_frame)), random_state=RANDOM_STATE)
scatter_payload = [
    {"age_jitter": _json_number(row["Age_jitter"]), "bmi": _json_number(row["BMI"]), "cluster_id": int(row["cluster_id"])}
    for _, row in plot_frame.iterrows()
]

dashboard_export = {
    "dataset": "CDC Diabetes Health Indicators — UCI ID 891",
    "target_name": target_col,
    "source_rows": int(len(df)),
    "source_columns": int(df.shape[1]),
    "sample_size": int(len(X_cluster)),
    "summary": {"sample_size": int(len(X_cluster)), "source_rows": int(len(df)), "source_columns": int(df.shape[1])},
    "feature_count": int(len(selected_features)),
    "selected_features": list(selected_features),
    "selected_model": str(best_model_name),
    "random_state": int(RANDOM_STATE),
    "metrics": metrics_payload,
    "cluster_sizes": cluster_sizes_payload,
    "cluster_profiles": cluster_profiles_payload,
    "overall_means": {feature: _json_number(overall_means[feature]) for feature in selected_features},
    "overall_stds": {feature: _json_number(overall_stds[feature]) for feature in selected_features},
    "scatter_points": scatter_payload,
    "runtime_versions": {"python": platform.python_version(), "scikit_learn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__},
}
with open("cluster_dashboard_data.json", "w", encoding="utf-8") as file:
    json.dump(dashboard_export, file, ensure_ascii=False, indent=2, allow_nan=False)

with open("requirements-model.txt", "w", encoding="utf-8") as file:
    file.write(
        "Flask>=3.0,<4.0\\n".replace("\\\\n", "\\n")
        + f"joblib=={joblib.__version__}\\n".replace("\\\\n", "\\n")
        + f"numpy=={np.__version__}\\n".replace("\\\\n", "\\n")
        + f"pandas=={pd.__version__}\\n".replace("\\\\n", "\\n")
        + f"scikit-learn=={sklearn.__version__}\\n".replace("\\\\n", "\\n")
        + "gunicorn>=21,<24\\n".replace("\\\\n", "\\n")
    )
print("Statistiques de l’application exportées : cluster_dashboard_data.json")
print(f"Modèle exporté : {best_model_name} ; {len(cluster_sizes_payload)} groupes/étiquettes affichés")
print("Versions du runtime :", dashboard_export["runtime_versions"])
print("Dépendances compatibles exportées : requirements-model.txt")
